# scorebook — first pass

**Findings go here, at the top, in plain English.** Not code comments — conclusions a
reader can understand without scrolling. Five sentences, one per question. Write them
last, put them first.

> _Nothing answered yet._

---

The five questions were committed to [`docs/questions.md`](../docs/questions.md) **before**
any analysis, along with what would falsify each one. Read that file before this one.

Loading and cleaning are already done by the package — see
[ADR 0006](../docs/decisions/0006-analysis-in-notebooks.md) for why the analysis is not.

In [ ]:
%matplotlib inline

import pandas as pd

from scorebook import clean, describe, plots
from scorebook.data import loaders

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

In [ ]:
# All 19 seasons. Downloads 6.8 MB the first time, then reads from ~/.cache/scorebook.
#
# Working offline? Swap in the committed sample instead:
#     raw = loaders.load_sample(Path("../data/sample_deliveries.csv"))
raw = loaders.load_deliveries()
deliveries = clean.prepare(raw)

print(describe.format_summary(describe.summarise(deliveries)))
deliveries.head()

## Before anything else: look at the nulls

A high null rate here almost always means "this event is rare", not "this data is
missing". `wicket_type` is 95% null because 95% of deliveries take no wicket. Reaching
for `dropna()` on this frame would delete nearly all of it.

See [ADR 0003](../docs/decisions/0003-informative-nulls.md).

In [ ]:
describe.null_profile(deliveries)

## One thing to know before you group anything

The text columns load as `category`, which cuts memory 7.4×. The cost is that a category
column keeps its full value list after a filter, and `groupby` defaults to
`observed=False` — so it emits one row per *category*, not per value present.

Filter to 2026 and group by team and you get 15 rows for the 10 teams that played, with
five defunct franchises credited with 0 runs. Run the cell below to see it.

**Always pass `observed=True`**, or call `clean.drop_unused_categories(frame)` after
filtering. See [README trap 4](../README.md) and
[the data dictionary](../docs/data-dictionary.md).

In [ ]:
recent = deliveries[deliveries.season_year == 2026]

played = recent["batting_team"].nunique()
unobserved = len(recent.groupby("batting_team", observed=False).size())
observed = len(recent.groupby("batting_team", observed=True).size())

print(f"teams that actually played in 2026: {played}")
print(f"rows from groupby(observed=False):  {unobserved}")
print(f"rows from groupby(observed=True):   {observed}")

# The phantom entries, credited with runs they never scored:
phantom = recent.groupby("batting_team", observed=False)["runs_off_bat"].sum()
phantom[phantom == 0]

## Q1 — Do runs per over spike in the death overs?

**Hypothesis:** yes, and the rise is steeper than the powerplay's.
**Falsified if:** the curve is flat after over 6, or the powerplay is the higher peak.

The total for a delivery is `runs_off_bat + extras` — `runs_off_bat` alone undercounts.
Note that a chase behaves differently from a first innings; consider splitting by
`innings`.

A *Manhattan* is the conventional chart here ([glossary](../docs/glossary.md)).

**Finding:**

**Caveat:**

## Q2 — Has scoring inflated across 19 seasons?

**Hypothesis:** risen, but less than commentary implies, and unevenly.
**Falsified if:** flat, or non-monotonic in a way no rule change explains.

Use `season_year`, not `season` — the label is a string and three values carry a slash
that no parsing rule handles correctly
([ADR 0004](../docs/decisions/0004-season-year.md)). Remember 2009 was played in South
Africa and 2020 in the UAE.

**Finding:**

**Caveat:**

## Q3 — Does a first-over wicket reduce the innings total?

**Hypothesis:** yes, but by under 10 runs on average.
**Falsified if:** the difference is under 2 runs, or reverses.

The hardest of the five, and the most transferable. "First-over wicket" is not a column:
group to innings level, flag whether any non-null `wicket_type` occurs where `over == 0`,
then join that flag back to compare totals.

**The trap:** innings that ended early — rain, or a chase completed — have low totals for
reasons unrelated to the wicket. Filter to innings of at least 19 overs first, and say so.

**Finding:**

**Caveat:**

## Q4 — Which venue shows the largest home advantage?

**Hypothesis:** a real but small effect, concentrated in two or three pitches.
**Falsified if:** no venue's advantage survives separating eras.

**This may not be answerable yet.** Match winners live in the 1,243 `_info.csv` files,
which are key-value long format and not loaded. Reporting this as deferred is a legitimate
outcome — write that down rather than forcing a weaker proxy.

Also: CSK and RR were suspended in 2016–17, and "home" is undefined for the 2009 and 2020
seasons played abroad.

**Finding:**

**Caveat:**

## Q5 — Are wides and no-balls getting rarer?

**Hypothesis:** slightly rarer, and smaller than the between-season noise.
**Falsified if:** rising, or the variance swamps the trend — in which case "cannot tell
from this data" is the honest answer.

`clean.fill_extras` already turned the nulls into zeros, which matters here: the rate needs
every delivery in its denominator, not only the ones that conceded an extra.

**Finding:**

**Caveat:**

## What didn't work

The section that makes the rest credible. An aggregation that double-counted, a hypothesis
that died, a confound with no available control.

An empty section here by the end means the questions were too safe — not that everything
worked.

---

When these are answered, copy the findings into
[`docs/results.md`](../docs/results.md) with the charts saved via
`plots.save_fig(figure, "name")`.